<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c07-header.png" alt="Nextia Learning · LLMs: How They Work and How to Choose One" width="100%">

# Solution: Measure practical performance

**[C07-M04-L03 · Measure practical performance](https://learning.nextia-ai.com/courses/llms/m04/measure-practical-performance/)** · C07, LLMs: How They Work and How to Choose One · Module 4, lesson 3 of 3 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

Summarise the recorded runs of eight models on Larkfield's 24 tickets: how many replies are ready to send, how long each
ticket takes (median and p95), what a ticket costs, and how sure a comparison on 24 tickets can be.

| | |
|---|---|
| **Time** | About 30 minutes |
| **You need** | A Google or Kaggle account, or Jupyter on your computer. No API key: every number comes from recorded runs. |
| **Before this** | [Score outputs consistently](https://learning.nextia-ai.com/courses/llms/m04/score-outputs-consistently/): the rubric and the ready rule |
| **Tested** | 2026-10-08, Python 3.14 with pandas 3.0.6, in Jupyter on macOS, from a fresh start |

### How to use this notebook

This notebook has every query, task and check of the lesson. The [lesson page](https://learning.nextia-ai.com/courses/llms/m04/measure-practical-performance/) has the full explanations, the diagrams and the knowledge checks that count toward your certificate. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your answer in the cell, run it, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/usmanahmed99/nextia-learning-labs/blob/main/C07/M04-L03-measure-practical-performance/measure-practical-performance-solution.ipynb).

## Setup: install pandas and get the recorded runs

Run the next cell. It installs pandas 3.0.6 and downloads 9 small files (under 1 MB): the rubric scores of all 192
replies and the recorded run of each model. It checks each file's SHA-256 checksum.

> **Check.** You should see `Ready: 192 scores, 8 runs, 24 tickets each.`

If a download fails, run the cell again. If it says a checksum is different, the file changed on the server: tell us.

In [ ]:
%pip install -q --disable-pip-version-check pandas==3.0.6
import hashlib, json, os, statistics, urllib.request
from pathlib import Path
import pandas as pd

RAW = "https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/C07/data/"
DATA = Path("c07-data")
SHA256 = {
    "scores.csv": "6e4b080b20344e9b9f8576f55035bc79c2742f5af53d1e1930752e03e67a7413",
    "runs/anthropic_claude-haiku-5-5.jsonl": "fffb250fffdd77601bfd753aecbae6cfa2ef54089caafc9c9f89cdc56929b2c1",
    "runs/anthropic_claude-sonnet-5-5.jsonl": "d44293dd3845f1e5903bf8c7b0f4eaa5b4f56c0b45bd70c6eeb85abf8ac303c9",
    "runs/anthropic_claude-opus-5-5.jsonl": "850c8a012831f07dbdff972732aa552e28f71862d01a3c119fae877e391a87fc",
    "runs/gemini_gemini-3.5-flash.jsonl": "1d61a9adf4b0b29d58f83164fbf27567a0d8f9a4659abbe2ff9f2e78e81f70fb",
    "runs/gemini_gemini-3.1-pro-preview.jsonl": "bcb8bcde60e11e9d8208f145c25a5ff754667c63083875a6d594309505947e7a",
    "runs/ollama_gemma3-4b.jsonl": "b1249db03ca14ccaabb8a50314adfc5c23d58d36df80cea2b6d617ad289be390",
    "runs/ollama_mistral-7b.jsonl": "8e0fdd9000a403843e489dc24c59f57764096d55a958f003545acff9cc6a3a4e",
    "runs/ollama_qwen3-4b-budget4000.jsonl": "a0a54e3bd3d3c60fb5607c7d08a55c2a8b9e2cb0e6bab3db8352fc25fb882a35"
}
for name, sha in SHA256.items():
    path = DATA / name
    local = os.environ.get("NX_C07_DATA")
    if not path.exists():
        path.parent.mkdir(parents=True, exist_ok=True)
        if local:
            path.write_bytes((Path(local) / name).read_bytes())
        else:
            urllib.request.urlretrieve(RAW + name, path)
    if hashlib.sha256(path.read_bytes()).hexdigest() != sha:
        raise SystemExit(f"{name}: the checksum is different. Delete the folder c07-data and run this cell again.")

scores = pd.read_csv(DATA / "scores.csv")
runs = {}
for f in sorted((DATA / "runs").glob("*.jsonl")):
    runs[f.stem] = pd.DataFrame([json.loads(line) for line in f.read_text(encoding="utf-8").splitlines()])
print(f"Ready: {len(scores)} scores, {len(runs)} runs, {len(next(iter(runs.values())))} tickets each.")

The next cell defines two small check helpers. You do not need to read them.

In [ ]:
def fingerprint(value):
    """A short hash of a value (floats rounded to 4 decimals), to check an answer without showing it."""
    def norm(v):
        if hasattr(v, "item"):
            v = v.item()
        if isinstance(v, float):
            return round(v, 4)
        if isinstance(v, (list, tuple)):
            return [norm(x) for x in v]
        return v
    return hashlib.sha256(json.dumps(norm(value)).encode()).hexdigest()[:16]


def expect(what, yours, expected_hash):
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
    elif fingerprint(yours) == expected_hash:
        print(f"Check passed: {what} is right.")
    else:
        print(f"Not yet: {what} is not the expected result. Read the task again, and check each step.")

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. Quality: the rubric scores

Each row of `scores` is one reply: the ticket, the model, the four rubric scores (0 to 2), the total and `ready`
(1 when correctness is 2 and no criterion is 0). Count the ready replies per model.

> **Check.** You should see 24 for four hosted models, 22 for Claude Sonnet 5.5, and 6, 12 and 14 for the local models.

In [ ]:
quality = scores.groupby("model_label")["ready"].sum().sort_values(ascending=False)
quality

## 2. Cost per ticket, on one ticket

Prices are per million tokens (standard API prices, checked on 2026-10-08; check the current pages before you use
them). Cost per ticket = input tokens × input price + (output tokens + reasoning tokens) × output price.
Claude counts its thinking inside `output_tokens`; Gemini reports its hidden reasoning in `reasoning_tokens`.

> **Predict.** Gemini 3.5 Flash has lower prices than Claude Sonnet 5.5. Which one costs more for ticket S01?

In [ ]:
PRICES = {  # US$ per million tokens (input, output), 2026-10-08
    "anthropic_claude-haiku-5-5": (0.10, 0.50),
    "anthropic_claude-sonnet-5-5": (2.00, 10.00),
    "anthropic_claude-opus-5-5": (4.00, 20.00),
    "gemini_gemini-3.5-flash": (1.50, 9.00),
    "gemini_gemini-3.1-pro-preview": (2.00, 12.00),
}

def cost(run, row):
    price_in, price_out = PRICES[run]
    reasoning = row.get("reasoning_tokens", 0) or 0
    return (row["input_tokens"] * price_in + (row["output_tokens"] + reasoning) * price_out) / 1_000_000

for run in ["anthropic_claude-sonnet-5-5", "gemini_gemini-3.5-flash"]:
    row = runs[run].set_index("id").loc["S01"]
    print(f"{run}: in {row['input_tokens']}, out {row['output_tokens']}, reasoning {row['reasoning_tokens']}, cost ${cost(run, row):.4f}")

> **Check.** Sonnet about $0.0030, Flash about $0.0101. Flash costs more, because of 933 hidden reasoning tokens.

## 3. Latency, tokens and cost for every model

The next cell builds the quality–latency–cost table of the lesson. `quantile(0.95)` uses linear interpolation, the
same method as the lesson (pandas' default).

In [ ]:
rows = []
for run, df in runs.items():
    label = scores.loc[scores["model"] == {"anthropic_claude-haiku-5-5": "haiku", "anthropic_claude-sonnet-5-5": "sonnet",
        "anthropic_claude-opus-5-5": "opus", "gemini_gemini-3.5-flash": "flash", "gemini_gemini-3.1-pro-preview": "pro",
        "ollama_gemma3-4b": "gemma", "ollama_mistral-7b": "mistral", "ollama_qwen3-4b-budget4000": "qwen"}[run], "model_label"].iloc[0]
    per_ticket = [cost(run, r) for _, r in df.iterrows()] if run in PRICES else None
    rows.append({
        "model": label,
        "ready": int(quality[label]),
        "median_s": round(df["seconds"].median(), 1),
        "p95_s": round(df["seconds"].quantile(0.95), 1),
        "median_out": int(df["output_tokens"].median()),
        "usd_per_1000": round(1000 * statistics.mean(per_ticket), 2) if per_ticket else None,
    })
table = pd.DataFrame(rows).sort_values("ready", ascending=False)
table

> **Check.** Claude Haiku 5.5: median 2.2 s, p95 3.2 s, $0.27 per 1,000 tickets. Gemini 3.1 Pro: median 9.8 s,
> p95 21.5 s, $14.83. Local models have no price (`None`): they run on your own hardware.

> **Your turn.** Store in `cost_haiku_s01` the cost of ticket S01 for Claude Haiku 5.5, in dollars (not rounded).
> Use the `cost()` function.

In [ ]:
cost_haiku_s01 = cost("anthropic_claude-haiku-5-5", runs["anthropic_claude-haiku-5-5"].set_index("id").loc["S01"])
cost_haiku_s01

In [ ]:
expect("cost_haiku_s01", cost_haiku_s01, 'aa63f092f4070e5c')

## 4. How sure are 24 tickets?

C03's interval, share ± 1.96 SE, fails for small samples near 100%. The Wilson interval below is the small-sample
version that the lesson uses. Then the paired difference of C03 *Compare alternatives*: only the tickets where the
two models disagree count.

In [ ]:
def wilson(k, n, z=1.96):
    p = k / n
    centre = (p + z * z / (2 * n)) / (1 + z * z / n)
    half = z * ((p * (1 - p) / n + z * z / (4 * n * n)) ** 0.5) / (1 + z * z / n)
    return round(100 * max(0, centre - half), 1), round(100 * min(1, centre + half), 1)


def paired(a, b):
    """Difference in ready replies (percentage points) of model a minus model b, on the same tickets, with its 95% interval."""
    ra = scores[scores["model"] == a].set_index("ticket")["ready"]
    rb = scores[scores["model"] == b].set_index("ticket")["ready"].loc[ra.index]
    n = len(ra)
    b_ = int(((ra == 1) & (rb == 0)).sum())
    c_ = int(((ra == 0) & (rb == 1)).sum())
    diff = (b_ - c_) / n
    se = ((b_ + c_ - (b_ - c_) ** 2 / n) ** 0.5) / n
    return b_, c_, round(100 * diff, 1), round(100 * (diff - 1.96 * se), 1), round(100 * (diff + 1.96 * se), 1)


print("Sonnet 22/24:", wilson(22, 24), " 24/24:", wilson(24, 24))
print("Sonnet - Opus:", paired("sonnet", "opus"))

> **Check.** Sonnet 74.2 to 97.7%; 24 of 24 gives 86.2 to 100%. Sonnet − Opus: 0 and 2 disagreements, −8.3 points,
> interval −19.4 to +2.7. The interval contains 0: the test cannot tell them apart.

> **Your turn.** Store in `mistral_minus_gemma` the result of `paired()` for Mistral 7B (`"mistral"`) minus Gemma 3 4B
> (`"gemma"`). Does the interval contain 0?

In [ ]:
mistral_minus_gemma = paired("mistral", "gemma")
mistral_minus_gemma   # (9, 3, 25.0, -1.5, 51.5): the interval just contains 0

In [ ]:
expect("mistral_minus_gemma", mistral_minus_gemma, 'd2f436a2eeab468c')

## 5. Change one thing: the volume

> **Your turn.** Larkfield's volume grows from 5,000 to 2,000,000 tickets a month. Compute the monthly cost of each
> hosted model from `table`, and write one sentence: does the cost still only break ties?

In [ ]:
monthly = table.dropna(subset=["usd_per_1000"]).assign(usd_per_month_5k=lambda t: t.usd_per_1000 * 5,
                                                     usd_per_month_2m=lambda t: t.usd_per_1000 * 2000)
monthly[["model", "ready", "usd_per_month_5k", "usd_per_month_2m"]]

> **Check.** At 5,000 tickets: about $1.35 (Haiku) to $74 (Gemini 3.1 Pro) a month. At 2 million: about $540 to
> $29,700. At the larger volume, cost becomes a real criterion.

## 6. Failure case: comparing prices instead of costs

A common mistake is to compare the price per token, or to forget the hidden reasoning tokens. The next cell computes
Gemini 3.5 Flash's cost per 1,000 tickets without the reasoning tokens.

In [ ]:
flash = runs["gemini_gemini-3.5-flash"]
without = 1000 * statistics.mean((r.input_tokens * 1.50 + r.output_tokens * 9.00) / 1e6 for r in flash.itertuples())
print(f"Without reasoning tokens: ${without:.2f} per 1,000 tickets; with them: "
      f"${table.set_index('model').loc['Gemini 3.5 Flash', 'usd_per_1000']:.2f}")

> **Check.** About $1.58 without and $10.90 with the reasoning tokens. The provider bills the hidden tokens, so the
> mistake makes Flash look seven times cheaper than it is.

## Recap

You built the quality–latency–cost table from recorded runs, found the cost per ticket (not the price per token),
read the median and the p95, and measured how sure a 24-ticket comparison can be. Next, Module 5 turns this evidence
into a decision with must-haves and weights.

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [Weighted selection](https://learning.nextia-ai.com/courses/llms/m05/weighted-selection/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/llms/m04/measure-practical-performance/)

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). The tickets are synthetic and contain no real customer data; the model replies are real recordings of 2026-10-08.